[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gps/gps.ipynb)

# Satellite Navigation

Satellite navigation locates a receiver and corrects its clock from the timestamps of arriving signals. Every transmission places the reception event on a light cone. Their intersection determines position and time together; uncertain timestamps spread that intersection into an uncertainty region whose shape depends on the satellite arrangement.

Projective spacetime algebra (STAP) holds events, light cones and measurement maps in `x+y+z+t-w0`. Its mixed signature supplies the causal geometry, and the null homogeneous direction carries translations. Light cones are quadratic correlations from events to planes; timing sensitivities build a separate positive information map whose inverse describes the uncertainty.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"
%matplotlib inline

In [ ]:
import numpy as np

from numga import Algebra, NumpyContext, stack, where
from examples.relativity.gps import render
from examples.util.animation import save_figure, show_animation

ga = Algebra("x+y+z+t-w0")
mv = NumpyContext(ga).multivector
Event = ga.gatype.antivector()
Slice = ga.gatype.vector()
Scalar = ga.gatype.scalar()
Readout = ga.gatype.from_blades("x y z t")
Step = Readout.dual().gatype.derive.output
Cone = ga.gatype((Slice, Event))
Information = ga.gatype((Readout, Step))
Covariance = ga.gatype((Step, Readout))
Orbit = ga.gatype.bivector()
# The position readouts, the planes across x, y and z.
Spatial = ga.gatype.from_blades("x y z")

origin = mv.w.dual()
across = mv.x.dual()
time = -mv.t.dual()


## Satellite signals

Six simulated satellites follow circular polar orbits at GPS altitude above a receiver at Earth's pole. Their timestamps have been corrected to GPS time; light propagation is modelled in flat spacetime. Lengths are in Earth radii and times in Earth-radius light-travel times, making the speed of light one. The orange surface marker is the receiver, and the blue segments connect it to the satellites at transmission.

In [ ]:
earth_radius_metres, orbit_altitude_metres = 6_371_000, 20_200_000
light_speed, gravitational_parameter = 299_792_458, 3.986004418e14
n_satellites, n_emission_steps = 6, 3
phase = np.deg2rad(20)
azimuths = np.arange(n_satellites) * (2 * np.pi / n_satellites)
phase_offsets = np.deg2rad([-12, -5, 8, 15, -9, 4])

# The planes share the polar axis; different phases give distinct elevations.
planes = (mv.yz * (-azimuths / 2)).exp() >> mv.xy
phases = mv.scalar((phase_offsets + phase)[..., None])
radius = (earth_radius_metres + orbit_altitude_metres) / earth_radius_metres
angular_speed = np.sqrt(gravitational_parameter / (earth_radius_metres + orbit_altitude_metres) ** 3)
angular_speed *= earth_radius_metres / light_speed

# Each snapshot uses its reception epoch as coordinate-time zero.
receiver = origin + across
initial_y, initial_z = 0.02, 0.01
coarse_position = receiver + mv.y.dual() * initial_y + mv.z.dual() * initial_z

In [ ]:
def orbit_positions(planes: Orbit, phases: Scalar, radius: float,
                    angular_speed: float, times: Scalar) -> Event:
    """Circular satellite trajectories, evaluated at coordinate times in the local inertial frame."""
    rotation = (-planes * ((phases + angular_speed * times) / 2)).exp()
    return origin + (rotation >> across) * radius + time * times


def spatial_distance(first: Event, second: Event) -> Scalar:
    """The distance between events of unit weight in the frame whose time `mv.t` reads: minus the squared norm of the
    line joining them, their interval, with the square of the time between them added back."""
    return ((mv.t & (first - second)).squared() - (first & second).scalar_norm_squared()).square_root()


def emitted_events(receiver: Event, planes: Orbit, phases: Scalar, radius: float,
                   angular_speed: float, n_steps: int) -> Event:
    """Retarded transmission events on the orbits whose signals reach the receiver together."""
    reception_time = mv.t & receiver
    emission_time = reception_time
    for _ in range(n_steps):
        satellites = orbit_positions(planes, phases, radius, angular_speed, emission_time)
        emission_time = reception_time - spatial_distance(receiver, satellites)
    return orbit_positions(planes, phases, radius, angular_speed, emission_time)


emitted = emitted_events(receiver, planes, phases, radius, angular_speed, n_emission_steps)
save_figure(render.draw_constellation(emitted, receiver, radius), "gps_constellation")

## Intersecting light cones

At an event on the cone, `cone(event)` is its tangent plane and `event & cone(event)` vanishes. The cone of a transmission is the line from it to an event, contracted with it, `emitted | (emitted & Event)`: the event pairs with that plane to the line's interval. Two cones of one Lorentz form differ by a pair of planes, the plane at infinity and their radical plane, so the differences of four cones give three radical planes, which meet in a line, and the fourth cone cuts that line in two events. Only one of them comes after every transmission: the reception.

In [ ]:
def light_cones(emitted: Event) -> Cone:
    """Homogeneous quadratic correlations whose vertices are the transmission events."""
    # The line from the transmission to an event, contracted with the transmission: a plane, which the event pairs
    # with to the line's interval, zero exactly when the line is a light ray.
    return emitted | (emitted & Event)


def intersect_cones(cones: Cone) -> Event:
    """The two intersections of four light cones, in a finite coordinate-time chart."""
    # Two cones of one Lorentz form differ by a pair of planes, the plane at infinity and their radical plane: the
    # difference sends an event to the radical plane times its weight plus the plane at infinity times the radical
    # plane's reading of it. Read at the origin, the second term is half the difference's own reading of the origin.
    difference = cones[1:] - cones[0]
    at_origin = difference(origin)
    radicals = at_origin - mv.w * ((origin & at_origin) / 2)
    axis = radicals[0] ^ radicals[1] ^ radicals[2]

    # Two time slices locate points on their common line. Normalizing by the
    # weight also gives these intersections their intrinsic event orientation.
    base = axis ^ mv.t
    base = base / (mv.w & base)
    tip = axis ^ (mv.t - mv.w)
    tip = tip / (mv.w & tip)
    direction = tip - base

    # The remaining cone cuts the line in two points.
    quadratic = direction & cones[0](direction)
    linear = base & cones[0](direction)
    constant = base & cones[0](base)
    root = (linear.squared() - quadratic * constant).square_root()
    parameters = stack((-linear - root, -linear + root)) / quadratic
    return base + direction * parameters


def reception(candidates: Event, emitted: Event) -> Event:
    """Of the cones' two intersections, the one every signal reaches after it left."""
    after = (mv.t & (candidates[:, None] - emitted)) > 0                  # [candidates, signals]
    return where(after[0].all(), candidates[0], candidates[1])


cones = light_cones(emitted[:4])
candidates = intersect_cones(cones)
located = reception(candidates, emitted[:4])
# The two intersections' distances from the coarse position, in kilometres, and their reception times, in
# milliseconds: only the later comes after every transmission.
candidate_distances = spatial_distance(candidates, coarse_position) * (earth_radius_metres / 1000)   # [candidates] Scalar+
candidate_times = (mv.t & candidates) * (earth_radius_metres / light_speed * 1000)                 # [candidates] Scalar+

In [ ]:
print("distances from the coarse position (km):", candidate_distances.to_array())
print("reception times (ms):", candidate_times.to_array())

## Timing noise and clock error

Noisy signals need not meet at one event. The fit asks each signal's interval, `receiver & cone(receiver)`, to vanish, weighed against how far its timestamp's uncertainty moves that interval: twice the light-travel time. A signal's readout of a small displacement of the reception is twice its cone's tangent plane there, `cone(receiver)`, and contributes a positive dyad to `information`; the sum measures which displacements the satellites can distinguish. Solving against that map corrects the event, and `information.inverse()` gives its position-and-time covariance.

In [ ]:
# The error of a range, set far above a real receiver's so that the uncertainty shows at the scale of the orbits,
# and the receiver clock's error.
range_error_metres, clock_error_seconds = 1_500_000, 30e-6
noise_seed, n_position_steps = 7, 5
sigma = range_error_metres / earth_radius_metres
clock_reading = mv.scalar([clock_error_seconds * light_speed / earth_radius_metres])
noise = mv.scalar(np.random.default_rng(noise_seed).normal(size=(n_satellites, 1)) * sigma)

# Timestamp errors perturb the measured cone vertices in time.
observed = emitted + time * noise
initial = coarse_position + time * clock_reading

In [ ]:
def intervals(cones: Cone, emitted: Event, receiver: Event) -> tuple[Scalar, Readout, Scalar]:
    """Each signal's interval from its transmission to a reception event, `receiver & cone(receiver)`, which vanishes
    on the light cone; its readout of a small displacement of the reception, twice the cone's tangent plane there;
    and how far a timestamp's error moves it, twice the light-travel time."""
    tangent = cones(receiver[..., None])
    # The tangent plane's part across the plane at infinity reads no displacement.
    readout = 2 * tangent.cast(Readout)
    lever = 2 * (mv.t & (receiver[..., None] - emitted))
    return receiver[..., None] & tangent, readout, lever


def locate(emitted: Event, initial: Event, sigma: float,
           n_steps: int) -> tuple[Event, Information, Covariance]:
    """Fit the reception event to noisy timestamps, each to its light cone, and recover the local uncertainty map."""
    cones = light_cones(emitted)
    receiver = initial
    for _ in range(n_steps):
        interval, readout, lever = intervals(cones, emitted, receiver)
        # Each signal's readout contributes a positive cost for the displacements it can see, weighted by how far
        # its timestamp's uncertainty moves its interval.
        weight = 1 / (lever * sigma) ** 2
        information = (weight * readout * (readout & Step)).sum(axis=-1)
        mismatch = (weight * readout * interval).sum(axis=-1)
        receiver = receiver - information.solve(mismatch)

    _, readout, lever = intervals(cones, emitted, receiver)
    information = (readout * (readout & Step) / (lever * sigma) ** 2).sum(axis=-1)
    covariance = information.inverse()
    return receiver, information, covariance


def position_spread(covariance: Covariance) -> Event:
    """The semi-axes of the position's one-standard-deviation ellipsoid: over the principal planes of the covariance
    between position readouts, each plane's covariance over its standard deviation."""
    variances, principal = (Spatial & covariance(Spatial)).eigh()
    return covariance(principal) / variances.square_root()


estimate, information, covariance = locate(observed, initial, sigma, n_position_steps)
semi_axes = position_spread(covariance)                                        # [3] Event-
save_figure(render.draw_solution(emitted, receiver, estimate, covariance, semi_axes, radius), "gps_positioning")
# The fit against the truth: the clock offset recovered, in microseconds, and its uncertainty, in nanoseconds; the
# position's error, and its principal standard deviations, the semi-axes' lengths, in metres.
clock_offset = (clock_reading - (mv.t & estimate)) * (earth_radius_metres / light_speed * 1e6)       # [] Scalar+
clock_spread = (mv.t & covariance(mv.t)).square_root() * (earth_radius_metres / light_speed * 1e9)   # [] Scalar+
position_error = spatial_distance(estimate, receiver) * earth_radius_metres                         # [] Scalar+
position_spreads = spatial_distance(origin + semi_axes, origin) * earth_radius_metres               # [3] Scalar+

In [ ]:
print("recovered clock offset (µs):", clock_offset.to_array())
print("clock uncertainty (ns):", clock_spread.to_array())
print("position error (m):", position_error.to_array())
print("position uncertainties along the principal axes (m):", position_spreads.to_array())

Orange marks the true position and blue the fitted one, and the blue surface about it is the position's one-standard-deviation ellipsoid, its principal radii the position's standard deviations. The timestamps' error is set far above a real receiver's, fifteen hundred kilometres of range, so that the ellipsoid shows at the scale of the orbits. The clock uncertainty remains part of the inverse information map, so the position ellipsoid includes the difficulty of separating location from clock error.

## Satellite geometry and uncertainty

Satellites crowded into one patch of sky leave similar changes in range when the receiver moves radially. A clock error can imitate much of that change. Spreading the satellites across the sky separates those readouts and shrinks the uncertainty. The view keeps its scale fixed, and each frame uses the same timestamp-noise draw. The Lorentz correlation sets the light cones throughout; the positive information correlation changes with the measurements.

In [ ]:
n_frames, duration_ms = 60, 65
first_phase, last_phase = np.deg2rad([5, 50])


def geometry_frames(n_frames: int):
    for phase in np.linspace(first_phase, last_phase, n_frames):
        phases = mv.scalar((phase_offsets + phase)[..., None])
        emitted = emitted_events(receiver, planes, phases, radius, angular_speed, n_emission_steps)
        observed = emitted + time * noise
        estimate, _, covariance = locate(observed, initial, sigma, n_position_steps)
        yield emitted, receiver, estimate, covariance, position_spread(covariance)


show_animation(render.animate_geometry(geometry_frames(n_frames), radius),
               "gps_uncertainty", duration_ms, n_frames=n_frames, forward_then_back=True)

In GPS positioning literature, [Bancroft's method](https://gssc.esa.int/navipedia/index.php/Bancroft_Method) gives an algebraic solution for four satellites, while [weighted least squares](https://gssc.esa.int/navipedia/index.php/Weighted_Least_Square_Solution_%28WLS%29) fits redundant measurements. The dependence of uncertainty on satellite arrangement is called [dilution of precision](https://gssc.esa.int/navipedia/index.php?title=Positioning_Error).